In [1]:
import polars as pl
from datasets import load_dataset

/home/lewis-work/miniconda3/envs/poly/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset(
    "SII-WANGZJ/Polymarket_data",
    data_files="quant.parquet",
    streaming=True,
    split="train"
)

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
# Grab first 1000 rows to inspect
sample = []
for i, row in enumerate(dataset):
    sample.append(row)
    if i >= 999:
        break

In [5]:
q_sample = pl.DataFrame(sample)

In [7]:
markets = pl.read_parquet('data/raw/markets.parquet')

In [ ]:
political_keywords = [
    "election", "president", "congress", "senate", "minister",
    "vote", "party", "democrat", "republican", "trump", "biden",
    "harris", "political", "govern", "parliament", "prime minister",
    "geopolit", "ukraine", "israel", "nato", "war", "sanction"
]

sports_noise = ["o/u", "spread", "over/under", "moneyline", 
                "vs.", "nfl", "nba", "mlb", "nhl", "epl"]
def political_search(df):
    # Apply filter using political words
    political = df.filter(
        pl.col('closed') == 1,
        pl.any_horizontal([
            pl.col('question').str.to_lowercase().str.contains(kw)
            for kw in political_keywords
        ])
    )

    # Remove sports noise which is missed in 1st filter
    political_clean = political.filter(
        ~pl.any_horizontal([
            pl.col('question').str.to_lowercase().str.contains(kw)
            for kw in sports_noise
        ])
    )

    # Parse outcome_prices to create a resolved_yes column
    def _parse_resolved_yes(x):
        import ast
        try:
            # Handle both JSON array strings and plain strings
            x_list = ast.literal_eval(x)
            return float(x_list[0]) > 0.5
        except Exception:
            return None

    # Apply parsing function to outcome_prices column
    politicalNO_final = political_clean.with_columns(
        pl.col('outcome_prices').map_elements(
            _parse_resolved_yes,
            return_dtype=pl.Boolean
        ).alias('resolved_yes')
    )
     

    return political_final

political_df = political_search(markets)

In [47]:
political_df.write_parquet('data/processed/markets_political.parquet')

In [49]:
political_df.shape

(20689, 21)

In [57]:
political_df.null_count() / len(political_df) * 100

id,question,slug,condition_id,token1,token2,answer1,answer2,closed,active,archived,outcome_prices,volume,event_id,event_slug,event_title,created_at,end_date,updated_at,neg_risk,resolved_yes
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009667


In [63]:
5952+14735

20687

In [53]:
print(political_df.filter(
    pl.col('resolved_yes').is_null()
))

shape: (2, 21)
┌─────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ id      ┆ question   ┆ slug       ┆ condition ┆ … ┆ end_date  ┆ updated_a ┆ neg_risk ┆ resolved_ │
│ ---     ┆ ---        ┆ ---        ┆ _id       ┆   ┆ ---       ┆ t         ┆ ---      ┆ yes       │
│ str     ┆ str        ┆ str        ┆ ---       ┆   ┆ datetime[ ┆ ---       ┆ u8       ┆ ---       │
│         ┆            ┆            ┆ str       ┆   ┆ ms, UTC]  ┆ datetime[ ┆          ┆ bool      │
│         ┆            ┆            ┆           ┆   ┆           ┆ ms, UTC]  ┆          ┆           │
╞═════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═══════════╡
│ 1488488 ┆ Israel     ┆ israel-fal ┆ 0x2a39e5c ┆ … ┆ 2026-04-3 ┆ 2026-03-0 ┆ 0        ┆ null      │
│         ┆ false flag ┆ se-flag-at ┆ 0288826ed ┆   ┆ 0         ┆ 3         ┆          ┆           │
│         ┆ attack     ┆ tack-confi ┆ d2c022cc9 ┆   ┆ 00:00:00  ┆ 03:45:17  

In [65]:
print(f"Ratio: {round(5952/20687, 3)}:{round(14735/20687, 3)}")

Ratio: 0.288:0.712


In [ ]:
political_df['end_date']

ModuleNotFoundError: No module named 'matplotlib'

In [55]:
print(political_df['resolved_yes'].value_counts())

shape: (3, 2)
┌──────────────┬───────┐
│ resolved_yes ┆ count │
│ ---          ┆ ---   │
│ bool         ┆ u32   │
╞══════════════╪═══════╡
│ true         ┆ 5952  │
│ null         ┆ 2     │
│ false        ┆ 14735 │
└──────────────┴───────┘


In [ ]:
def _parse_resolved_yes(x):
    try:
        # Handle both JSON array strings and plain strings
        if x is None:
            return None
        x = x.strip()
        # Try standard JSON parse first
    

        parsed = json.loads(x)
        return float(parsed[0]) > 0.5
    except Exception:
        try:
            # Handle single-quoted or malformed strings
            x = x.replace("'", '"')
            parsed = json.loads(x)
            return float(parsed[0]) > 0.5
        except Exception:
            return None   

In [39]:
import ast
l = political_df['outcome_prices'].to_list()
parsed_list = [ast.literal_eval(item) if isinstance(item, str) else item for item in l]

In [44]:
ast.literal_eval(l[0])

['1', '0']

In [36]:
for x in political_df['outcome_prices'].to_list():
    print(x)

['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['0', '1']
['1', '0']
['0', '1']
['0', '1']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['0', '1']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['1', '0']
['1', '0']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['0', '1']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['0', '1']
['0', '1']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['1', '0']
['0', '1']
['0', '1']
['1', '0']
['0', '1']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['1', '0']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['1', '0']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']
['0', '1']

In [10]:
markets.filter(
    pl.col('closed') == 1
)

id,question,slug,condition_id,token1,token2,answer1,answer2,closed,active,archived,outcome_prices,volume,event_id,event_slug,event_title,created_at,end_date,updated_at,neg_risk
str,str,str,str,str,str,str,str,u8,u8,u8,str,f64,str,str,str,"datetime[ms, UTC]","datetime[ms, UTC]","datetime[ms, UTC]",u8
"""944245""","""Bitcoin Up or Down - December …","""btc-updown-15m-1765927800""","""0xf34569d69142dd4d0ae07416061d…","""670014420876296989998962834629…","""870209324516362342907883186753…","""Up""","""Down""",1,1,0,"""['1', '0']""",103082.899091,"""106225""","""btc-updown-15m-1765927800""","""Bitcoin Up or Down - December …",2025-12-15 23:32:31 UTC,2025-12-16 23:45:00 UTC,2026-02-17 14:47:15 UTC,0
"""944246""","""Solana Up or Down - December 1…","""sol-updown-15m-1765927800""","""0xb2507c7f19be7dfcf3c66299439b…","""726309663230036633043625744393…","""376900909309768613852881266039…","""Up""","""Down""",1,1,0,"""['1', '0']""",7967.175044,"""106226""","""sol-updown-15m-1765927800""","""Solana Up or Down - December 1…",2025-12-15 23:32:31 UTC,2025-12-16 23:45:00 UTC,2026-02-17 14:47:15 UTC,0
"""944247""","""Ethereum Up or Down - December…","""eth-updown-15m-1765927800""","""0x77cde2af1ac034348adab9d8de8c…","""246424976012767732853248836185…","""101017665224423762830049503255…","""Up""","""Down""",1,1,0,"""['1', '0']""",11031.484996,"""106224""","""eth-updown-15m-1765927800""","""Ethereum Up or Down - December…",2025-12-15 23:32:31 UTC,2025-12-16 23:45:00 UTC,2026-02-17 14:47:15 UTC,0
"""944248""","""XRP Up or Down - December 16, …","""xrp-updown-15m-1765927800""","""0x56d4faf8161c8ec8655864e081bd…","""369683653270198554522976839189…","""202090247490892321898712987862…","""Up""","""Down""",1,1,0,"""['1', '0']""",5191.069866,"""106227""","""xrp-updown-15m-1765927800""","""XRP Up or Down - December 16, …",2025-12-15 23:32:34 UTC,2025-12-16 23:45:00 UTC,2026-02-17 14:47:15 UTC,0
"""944321""","""XRP Up or Down - December 16, …","""xrp-updown-15m-1765928700""","""0x5481141979ef8c3b76084469f162…","""105494416864949210663739753390…","""403839591472131780740459614415…","""Up""","""Down""",1,1,0,"""['1', '0']""",8359.971574,"""106238""","""xrp-updown-15m-1765928700""","""XRP Up or Down - December 16, …",2025-12-15 23:47:31 UTC,2025-12-17 00:00:00 UTC,2026-02-17 14:47:15 UTC,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""673418""","""Solana Up or Down - November 9…","""sol-updown-15m-1762740900""","""0x437d56f248586cd8117f25941764…","""112254108055325679807167284370…","""184119611776106036431313907771…","""Up""","""Down""",1,1,0,"""['0', '1']""",868.638209,"""77271""","""sol-updown-15m-1762740900""","""Solana Up or Down - November 9…",2025-11-09 23:16:53 UTC,2025-11-10 02:30:00 UTC,2026-02-17 14:47:15 UTC,0
"""673419""","""XRP Up or Down - November 9, 9…","""xrp-updown-15m-1762740900""","""0x2476579cdb3616a508dcfa884165…","""540883782187976465472151186464…","""416452424805185385930775710578…","""Up""","""Down""",1,1,0,"""['1', '0']""",13563.832898,"""77270""","""xrp-updown-15m-1762740900""","""XRP Up or Down - November 9, 9…",2025-11-09 23:16:53 UTC,2025-11-10 02:30:00 UTC,2026-02-17 14:47:15 UTC,0
"""673420""","""Bitcoin Up or Down - November …","""btc-updown-15m-1762740900""","""0xfe1319b4d432ff46bb3484669bdd…","""113372711562495889610683295427…","""915940690094051435197751726339…","""Up""","""Down""",1,1,0,"""['0', '1']""",76070.488629,"""77272""","""btc-updown-15m-1762740900""","""Bitcoin Up or Down - November …",2025-11-09 23:16:55 UTC,2025-11-10 02:30:00 UTC,2026-02-17 14:47:15 UTC,0


In [6]:
q_sample

timestamp,block_number,transaction_hash,log_index,market_id,condition_id,event_id,price,usd_amount,token_amount,side,maker,taker
i64,i64,str,i64,str,str,str,f64,f64,f64,str,str,str
1669060169,35896869,"""05a7f9b0b016d7e431b4e62b374870…",204,"""240380""","""0x41190eb9336ae73949c04f4900f9…","""5828""",0.5,50.0,100.0,"""SELL""","""0xEA5981CA48Dc40C950fC1B2496c4…","""0xEA5981CA48Dc40C950fC1B2496c4…"
1669060373,35896929,"""3a5412c9d8129e8a27ddb43cdf60ad…",161,"""240380""","""0x41190eb9336ae73949c04f4900f9…","""5828""",0.6,60.0,100.0,"""SELL""","""0xEA5981CA48Dc40C950fC1B2496c4…","""0xEA5981CA48Dc40C950fC1B2496c4…"
1669060582,35897058,"""4541524b5b2f910c9f3f5d12feae26…",189,"""240380""","""0x41190eb9336ae73949c04f4900f9…","""5828""",0.5,5.0,10.0,"""SELL""","""0xEA5981CA48Dc40C950fC1B2496c4…","""0xEA5981CA48Dc40C950fC1B2496c4…"
1669060582,35897058,"""b6dadd061a1b4fa4a46be4f235db01…",180,"""240380""","""0x41190eb9336ae73949c04f4900f9…","""5828""",0.5,5.0,10.0,"""SELL""","""0xEA5981CA48Dc40C950fC1B2496c4…","""0xEA5981CA48Dc40C950fC1B2496c4…"
1669060582,35897058,"""e444cffd294164908e4862b1859ea5…",170,"""240380""","""0x41190eb9336ae73949c04f4900f9…","""5828""",0.5,5.0,10.0,"""SELL""","""0xEA5981CA48Dc40C950fC1B2496c4…","""0xEA5981CA48Dc40C950fC1B2496c4…"
…,…,…,…,…,…,…,…,…,…,…,…,…
1671369633,36979995,"""24a7260685dec6e852b7d4018a43bd…",67,"""248205""","""0x1e7db4f6ca3919aa41887f970160…","""868350""",0.45,122.22,222.22,"""BUY""","""0x8d0328E8aEaa03Bb13796c0EB312…","""0x861AEE254635B06FDADb884CB609…"
1671369789,36980026,"""612b84a05b58ffcb5262fd20252a82…",99,"""248205""","""0x1e7db4f6ca3919aa41887f970160…","""868350""",0.45,53.91,98.02,"""BUY""","""0x8d0328E8aEaa03Bb13796c0EB312…","""0x7Dc032f3Aa0E0715001342966758…"
1671369993,36980192,"""2a28abcb5a228f57326aa12f904d85…",160,"""248205""","""0x1e7db4f6ca3919aa41887f970160…","""868350""",0.45,61.11,111.11,"""BUY""","""0x8d0328E8aEaa03Bb13796c0EB312…","""0x0CD96Ee3b364b86Cb07c86246Af9…"
